# Fill-in-the-Blank (FITB) Generator

Schema-driven · Batch-based · Judge-validated

Each FITB question has:
- `sentence_with_blank` — a sentence containing `_______`
- `options` — exactly 4 choices (one correct, three plausible distractors)
- `correct_answer` — the word/phrase that fills the blank
- `explanation` — why the correct answer fits

**CEFR sub-types:**
| Level | Task type |
|---|---|
| A1/A2 | Basic vocabulary and simple grammar (verbs, articles, prepositions) |
| B1/B2 | Phrasal verbs, collocations, complex tenses |
| C1/C2 | Advanced vocabulary, idiomatic expressions, academic language |

**Output:** `data/fitb/fitb_generator_output.json`

In [ ]:
from __future__ import annotations
import json, os, re, sys
from pathlib import Path
from typing import Literal, Optional

import dspy
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

from utils import configure_lm, load_json, save_json
print('Project root:', PROJECT_ROOT)

In [ ]:
# ── Language model setup ──────────────────────────────────────────────────────
lm = configure_lm()   # reads MISTRAL_API_KEY / model from .env + dspy.config.yaml
dspy.configure(lm=lm)
print('LM:', lm.model)

In [ ]:
# ── Artifact paths ────────────────────────────────────────────────────────────
ARTIFACTS = PROJECT_ROOT / 'artifacts'

DIFFICULTY_ARTIFACT = ARTIFACTS / 'simple_difficulty_optimized.json'
RUBRIC_ARTIFACT     = ARTIFACTS / 'rubric_agent_optimized.json'

OUTPUT_PATH = PROJECT_ROOT / 'data' / 'fitb' / 'fitb_generator_output.json'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print('Difficulty artifact:', DIFFICULTY_ARTIFACT.exists())
print('Rubric artifact    :', RUBRIC_ARTIFACT.exists())
print('Output path        :', OUTPUT_PATH)

In [ ]:
# ── Input schema models ───────────────────────────────────────────────────────
class SubtopicRequirement(BaseModel):
    '''Per-subtopic question counts for each of the 6 CEFR levels.'''
    subtopic:  str
    a1_count:  int = 0
    a2_count:  int = 0
    b1_count:  int = 0
    b2_count:  int = 0
    c1_count:  int = 0
    c2_count:  int = 0

    @property
    def easy_count(self) -> int:   return self.a1_count + self.a2_count
    @property
    def medium_count(self) -> int: return self.b1_count + self.b2_count
    @property
    def hard_count(self) -> int:   return self.c1_count + self.c2_count
    @property
    def total(self) -> int:        return self.easy_count + self.medium_count + self.hard_count


class GenerationConstraints(BaseModel):
    questions_per_iteration: int = 3
    max_iterations:          int = 10
    difficulty_match:        bool = True
    rubric_min_score:        float = 0.7


class InputSchema(BaseModel):
    topic:        str
    subtopics:    list[SubtopicRequirement]
    constraints:  GenerationConstraints = Field(default_factory=GenerationConstraints)

In [ ]:
# ── Question item & store ─────────────────────────────────────────────────────
class FITBItem(BaseModel):
    question_number:    int
    topic:              str
    subtopic:           str | None
    target_cefr:        str
    target_difficulty:  str
    instruction:        str
    sentence_with_blank: str   # contains _______ as the blank marker
    options:            list[str]   # exactly 4
    correct_answer:     str
    explanation:        str


class FITBQuestionStore(BaseModel):
    easy:   list[FITBItem] = Field(default_factory=list)
    medium: list[FITBItem] = Field(default_factory=list)
    hard:   list[FITBItem] = Field(default_factory=list)

    def add(self, item: FITBItem) -> None:
        d = item.target_difficulty.strip().lower()
        if d == 'easy':   self.easy.append(item)
        elif d == 'medium': self.medium.append(item)
        elif d == 'hard':   self.hard.append(item)

    def count(self, difficulty: str) -> int:
        return len(getattr(self, difficulty.lower(), []))

    @property
    def total(self) -> int:
        return len(self.easy) + len(self.medium) + len(self.hard)

In [ ]:
# ── Generation models + DSPy Signature ───────────────────────────────────────
class ExampleFITBQuestion(BaseModel):
    instruction:         str
    sentence_with_blank: str
    options:             list[str]
    correct_answer:      str
    explanation:         str
    difficulty:          str
    cefr:                str
    subtopic:            str


class FITBGenerationRequest(BaseModel):
    topic:               str
    subtopic:            str
    target_cefr:         str
    target_difficulty:   str
    example_questions:   list[ExampleFITBQuestion]
    already_used_sentences: list[str]
    batch_size:          int


class FITBGeneratedQuestion(BaseModel):
    instruction:         str
    sentence_with_blank: str   # must contain _______
    options:             list[str]   # exactly 4
    correct_answer:      str
    explanation:         str


class FITBGenerationBatch(BaseModel):
    questions: list[FITBGeneratedQuestion]


class FITBGeneratorSignature(dspy.Signature):
    '''Generate a batch of fill-in-the-blank (FITB) questions for English language learners.

    Rules:
    - sentence_with_blank must contain exactly one _______ (7 underscores) as the blank
    - options: exactly 4 choices — one correct, three plausible but wrong distractors
    - correct_answer must exactly match one of the four options
    - Distractors must be grammatically plausible but semantically or grammatically incorrect
    - A1/A2: basic vocabulary, verb conjugation, articles, prepositions
    - B1/B2: phrasal verbs, collocations, modal verbs, tense distinctions
    - C1/C2: idiomatic expressions, academic vocabulary, subtle semantic distinctions
    - instruction: must describe the task clearly (e.g. "Choose the word that best completes the sentence.")
    '''
    request: FITBGenerationRequest = dspy.InputField(desc='Generation parameters and examples')
    batch:   FITBGenerationBatch   = dspy.OutputField(desc='Generated FITB questions')

In [ ]:
# ── Difficulty Judge ──────────────────────────────────────────────────────────
class DifficultyResult(BaseModel):
    question_id:          str
    predicted_cefr:       Literal['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
    predicted_difficulty: Literal['Easy', 'Medium', 'Hard']

class DifficultyOutput(BaseModel):
    results: list[DifficultyResult]


class FITBDifficultySignature(dspy.Signature):
    '''Classify FITB questions by CEFR level and difficulty.
    Analyse: vocabulary of the blank word, distractor sophistication,
    sentence complexity, and grammatical concepts tested.
    A1/A2=Easy, B1/B2=Medium, C1/C2=Hard.
    '''
    questions: str            = dspy.InputField(desc='JSON list of FITB questions with question_id')
    output:    DifficultyOutput = dspy.OutputField(desc='Difficulty classifications')


class DifficultyJudgeWrapper:
    def __init__(self, artifact_path: Path):
        self.judge = dspy.Predict(FITBDifficultySignature)
        if artifact_path.exists():
            self.judge.load(str(artifact_path))
            print(f'  DifficultyJudge loaded from {artifact_path.name}')
        else:
            print('  DifficultyJudge using zero-shot (no artifact)')

    def judge_batch(self, questions: list[dict]) -> list[DifficultyResult]:
        result = self.judge(questions=json.dumps(questions))
        return result.output.results

In [ ]:
# ── Rubric Judge ──────────────────────────────────────────────────────────────
class RubricScore(BaseModel):
    question_id:           str
    grammatical_accuracy:  float = Field(ge=0, le=1)
    spelling:              float = Field(ge=0, le=1)
    ambiguity:             float = Field(ge=0, le=1)
    distractor_quality:    float = Field(ge=0, le=1)   # plausible but clearly wrong
    instruction_clarity:   float = Field(ge=0, le=1)
    option_consistency:    float = Field(ge=0, le=1)
    readability:           float = Field(ge=0, le=1)
    avg_score:             float = Field(ge=0, le=1)

class RubricOutput(BaseModel):
    scores: list[RubricScore]


class FITBRubricSignature(dspy.Signature):
    '''Score FITB question quality on 7 criteria (0-1 each).
    grammatical_accuracy: sentence is grammatically correct with the blank filled
    spelling: no spelling errors in sentence or options
    ambiguity: blank has only one defensible correct answer
    distractor_quality: wrong options are plausible but clearly incorrect
    instruction_clarity: task description is unambiguous
    option_consistency: all options are same part of speech / grammatical form
    readability: sentence is natural and appropriate for the CEFR level
    avg_score: mean of the seven scores above.
    '''
    questions: str         = dspy.InputField(desc='JSON list of FITB questions with question_id')
    output:    RubricOutput = dspy.OutputField(desc='Quality scores per question')


class RubricJudgeWrapper:
    def __init__(self, artifact_path: Path):
        self.judge = dspy.Predict(FITBRubricSignature)
        if artifact_path.exists():
            self.judge.load(str(artifact_path))
            print(f'  RubricJudge loaded from {artifact_path.name}')
        else:
            print('  RubricJudge using zero-shot (no artifact)')

    def score_batch(self, questions: list[dict]) -> list[RubricScore]:
        result = self.judge(questions=json.dumps(questions))
        return result.output.scores

In [ ]:
# ── Hard validation ───────────────────────────────────────────────────────────
def hard_validate(q: FITBGeneratedQuestion) -> tuple[bool, str]:
    if '_______' not in q.sentence_with_blank:
        return False, 'sentence_with_blank missing _______ marker'
    if len(q.options) != 4:
        return False, f'Expected 4 options, got {len(q.options)}'
    if q.correct_answer not in q.options:
        return False, f'correct_answer "{q.correct_answer}" not in options'
    if len(set(q.options)) != 4:
        return False, 'Duplicate options found'
    return True, ''

In [ ]:
# ── Generator agent ───────────────────────────────────────────────────────────
class FITBGeneratorAgent(dspy.Module):
    def __init__(self, *, store: FITBQuestionStore,
                 difficulty_judge: DifficultyJudgeWrapper,
                 rubric_judge: RubricJudgeWrapper):
        super().__init__()
        self.store            = store
        self.difficulty_judge = difficulty_judge
        self.rubric_judge     = rubric_judge
        self.generator        = dspy.Predict(FITBGeneratorSignature)
        self._question_counter = 0

    def forward(self, *, schema: InputSchema, example_questions: list[ExampleFITBQuestion],
                topic: str, subtopic: str, target_cefr: str, target_difficulty: str,
                target_count: int, rejected: list, warnings: list) -> None:

        used_sentences = [q.sentence_with_blank for d in (self.store.easy, self.store.medium, self.store.hard)
                          for q in d]
        iteration = 0
        difficulty_map = {'A1':'Easy','A2':'Easy','B1':'Medium','B2':'Medium','C1':'Hard','C2':'Hard'}

        while self.store.count(target_difficulty) < target_count and iteration < schema.constraints.max_iterations:
            iteration += 1
            needed = target_count - self.store.count(target_difficulty)
            batch_size = min(schema.constraints.questions_per_iteration, needed + 2)

            req = FITBGenerationRequest(
                topic=topic, subtopic=subtopic,
                target_cefr=target_cefr, target_difficulty=target_difficulty,
                example_questions=example_questions,
                already_used_sentences=used_sentences,
                batch_size=batch_size,
            )
            try:
                result = self.generator(request=req)
                candidates = result.batch.questions
            except Exception as e:
                warnings.append(f'[{target_cefr}] Generation error iter {iteration}: {e}')
                continue

            # Hard validate
            valid = []
            for q in candidates:
                ok, reason = hard_validate(q)
                if ok:
                    valid.append(q)
                else:
                    rejected.append({'cefr': target_cefr, 'reason': reason, 'q': q.sentence_with_blank[:60]})

            if not valid:
                continue

            # Difficulty judge
            judge_input = [{'question_id': f'q{i}', 'instruction': q.instruction,
                            'sentence_with_blank': q.sentence_with_blank,
                            'options': q.options, 'correct_answer': q.correct_answer,
                            'target_cefr': target_cefr, 'target_difficulty': target_difficulty}
                           for i, q in enumerate(valid)]
            try:
                diff_results = {r.question_id: r for r in self.difficulty_judge.judge_batch(judge_input)}
            except Exception as e:
                warnings.append(f'[{target_cefr}] Difficulty judge error: {e}')
                diff_results = {}

            # Rubric judge
            try:
                rubric_results = {r.question_id: r for r in self.rubric_judge.score_batch(judge_input)}
            except Exception as e:
                warnings.append(f'[{target_cefr}] Rubric judge error: {e}')
                rubric_results = {}

            # Accept/reject
            for i, q in enumerate(valid):
                qid = f'q{i}'
                dr  = diff_results.get(qid)
                rr  = rubric_results.get(qid)

                if dr and difficulty_map.get(dr.predicted_cefr) != target_difficulty:
                    rejected.append({'cefr': target_cefr, 'reason': f'Difficulty mismatch: predicted {dr.predicted_cefr}', 'q': q.sentence_with_blank[:60]})
                    continue
                if rr and rr.avg_score < schema.constraints.rubric_min_score:
                    rejected.append({'cefr': target_cefr, 'reason': f'Rubric too low: {rr.avg_score:.2f}', 'q': q.sentence_with_blank[:60]})
                    continue

                self._question_counter += 1
                item = FITBItem(
                    question_number=self._question_counter,
                    topic=topic, subtopic=subtopic,
                    target_cefr=target_cefr, target_difficulty=target_difficulty,
                    instruction=q.instruction, sentence_with_blank=q.sentence_with_blank,
                    options=q.options, correct_answer=q.correct_answer, explanation=q.explanation,
                )
                self.store.add(item)
                used_sentences.append(q.sentence_with_blank)

                if self.store.count(target_difficulty) >= target_count:
                    break

In [ ]:
# ── Example questions per CEFR ────────────────────────────────────────────────
EXAMPLE_QUESTIONS: list[ExampleFITBQuestion] = [
    ExampleFITBQuestion(
        instruction='Choose the word that best completes the sentence.',
        sentence_with_blank='She _______ to school every day.',
        options=['go', 'goes', 'going', 'gone'],
        correct_answer='goes',
        explanation='Third-person singular present simple requires -s/-es.',
        difficulty='Easy', cefr='A1', subtopic='present simple'
    ),
    ExampleFITBQuestion(
        instruction='Choose the word that best completes the sentence.',
        sentence_with_blank='I have been studying for _______ hours.',
        options=['few', 'a few', 'little', 'a little'],
        correct_answer='a few',
        explanation='"A few" is used with countable nouns like "hours".',
        difficulty='Easy', cefr='A2', subtopic='quantifiers'
    ),
    ExampleFITBQuestion(
        instruction='Choose the word or phrase that best completes the sentence.',
        sentence_with_blank='She managed to _______ the problem by staying calm.',
        options=['deal with', 'deal in', 'deal for', 'deal on'],
        correct_answer='deal with',
        explanation='"Deal with" means to handle or manage a situation.',
        difficulty='Medium', cefr='B1', subtopic='phrasal verbs'
    ),
    ExampleFITBQuestion(
        instruction='Choose the word or phrase that best completes the sentence.',
        sentence_with_blank='The new policy is _______ effect from next Monday.',
        options=['in', 'into', 'at', 'on'],
        correct_answer='in',
        explanation='"In effect" is a fixed expression meaning currently operating.',
        difficulty='Medium', cefr='B2', subtopic='collocations'
    ),
    ExampleFITBQuestion(
        instruction='Choose the word that best completes the sentence.',
        sentence_with_blank='The author\'s argument, though _______, fails to address the core issue.',
        options=['compelling', 'repelling', 'dispelling', 'propelling'],
        correct_answer='compelling',
        explanation='"Compelling" means persuasive or convincing, fitting the academic register.',
        difficulty='Hard', cefr='C1', subtopic='academic vocabulary'
    ),
    ExampleFITBQuestion(
        instruction='Choose the word that best completes the sentence.',
        sentence_with_blank='His _______ remarks at the ceremony left a lasting impression on all present.',
        options=['eloquent', 'eloquence', 'eloquently', 'eloquentness'],
        correct_answer='eloquent',
        explanation='An adjective is needed before "remarks". "Eloquent" means fluent and persuasive.',
        difficulty='Hard', cefr='C2', subtopic='advanced adjectives'
    ),
]

In [ ]:
# ── Orchestrator ──────────────────────────────────────────────────────────────
_CEFR_TO_DIFFICULTY = {'A1':'Easy','A2':'Easy','B1':'Medium','B2':'Medium','C1':'Hard','C2':'Hard'}
_CEFR_LEVELS = ['A1','A2','B1','B2','C1','C2']

def run_fitb_generation(schema: InputSchema) -> tuple[FITBQuestionStore, list, list]:
    difficulty_judge = DifficultyJudgeWrapper(DIFFICULTY_ARTIFACT)
    rubric_judge     = RubricJudgeWrapper(RUBRIC_ARTIFACT)

    store    = FITBQuestionStore()
    rejected = []
    warnings = []

    agent = FITBGeneratorAgent(store=store, difficulty_judge=difficulty_judge, rubric_judge=rubric_judge)

    for subtopic_req in schema.subtopics:
        for cefr in _CEFR_LEVELS:
            count_attr = f'{cefr.lower()}_count'
            target_count = getattr(subtopic_req, count_attr, 0)
            if target_count == 0:
                continue
            difficulty = _CEFR_TO_DIFFICULTY[cefr]
            examples   = [q for q in EXAMPLE_QUESTIONS if q.cefr == cefr]
            print(f'  Generating {target_count}× {cefr} ({difficulty}) — {subtopic_req.subtopic}')
            agent.forward(
                schema=schema, example_questions=examples,
                topic=schema.topic, subtopic=subtopic_req.subtopic,
                target_cefr=cefr, target_difficulty=difficulty,
                target_count=target_count + store.count(difficulty),
                rejected=rejected, warnings=warnings,
            )

    return store, rejected, warnings

In [ ]:
# ── Define schema and run ─────────────────────────────────────────────────────
SCHEMA = InputSchema(
    topic='English Grammar and Vocabulary',
    subtopics=[
        SubtopicRequirement(subtopic='verb tenses',        a1_count=1, a2_count=1),
        SubtopicRequirement(subtopic='phrasal verbs',      b1_count=1, b2_count=1),
        SubtopicRequirement(subtopic='advanced vocabulary', c1_count=1, c2_count=1),
    ],
    constraints=GenerationConstraints(questions_per_iteration=3, max_iterations=8, rubric_min_score=0.65),
)

print(f'Schema: {SCHEMA.topic}')
for s in SCHEMA.subtopics:
    print(f'  {s.subtopic}: easy={s.easy_count} medium={s.medium_count} hard={s.hard_count} (total={s.total})')

print()
store, rejected, warnings = run_fitb_generation(SCHEMA)
print(f'\nGenerated: easy={len(store.easy)} medium={len(store.medium)} hard={len(store.hard)} total={store.total}')
print(f'Rejected : {len(rejected)}')
if warnings:
    for w in warnings: print(' ', w)

In [ ]:
# ── Display results ───────────────────────────────────────────────────────────
for difficulty in ('easy', 'medium', 'hard'):
    items = getattr(store, difficulty)
    if not items: continue
    print(f'\n{"="*70}')
    print(f'  {difficulty.upper()} ({len(items)} questions)')
    print(f'{"="*70}')
    for item in items:
        print(f'Q{item.question_number} [{item.target_cefr}] {item.subtopic}')
        print(f'  {item.instruction}')
        print(f'  {item.sentence_with_blank}')
        for j, opt in enumerate(item.options):
            tick = ' *' if opt == item.correct_answer else ''
            print(f'    {chr(65+j)}) {opt}{tick}')
        print(f'  Explanation: {item.explanation}')
        print()

In [ ]:
# ── Save output ───────────────────────────────────────────────────────────────
output = {
    'metadata': {
        'topic': SCHEMA.topic,
        'question_type': 'fitb',
        'total': store.total,
        'rejected': len(rejected),
    },
    'questions': {
        'easy':   [q.model_dump() for q in store.easy],
        'medium': [q.model_dump() for q in store.medium],
        'hard':   [q.model_dump() for q in store.hard],
    },
    'rejected': rejected,
}
OUTPUT_PATH.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved {store.total} questions to {OUTPUT_PATH}')